# 05 — Model Architecture & Implementation

**Project:** Early Disease Risk Predictor  
**Phase:** 3 — Model Architecture & Implementation  
**Framework:** Scikit-Learn + XGBoost  

---

This notebook defines every model that will be trained in Phase 4.  
No fitting or evaluation is performed here — this phase is concerned solely with:

1. **Framework selection** and justification  
2. **Architecture design** — layers, objectives, loss functions, and optimisers  
3. **Hyperparameter documentation** — full search space and final justified selections  
4. **Serialising model configurations** so Phase 4 can load them directly

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib

from sklearn.linear_model   import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neural_network  import MLPClassifier
from xgboost                 import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROC_DIR  = os.path.join('..', 'data', 'processed')
FIGS_DIR  = os.path.join('..', 'reports', 'figures')
MODEL_DIR = os.path.join('..', 'data', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

DISEASES     = ['diabetes', 'heart_disease', 'hypertension']
RANDOM_STATE = 42
TEST_SIZE    = 0.20
CV_FOLDS     = 5

print('Setup complete.')

## 1. Dataset Summary

Load the processed feature matrices to inspect shapes and class balance.  
This information directly informs architectural decisions (e.g. `scale_pos_weight`).

In [ ]:
summary_rows = []
for disease in DISEASES:
    X = pd.read_csv(os.path.join(PROC_DIR, f'X_{disease}.csv'))
    y = pd.read_csv(os.path.join(PROC_DIR, f'y_{disease}.csv')).squeeze()
    n_neg  = int((y == 0).sum())
    n_pos  = int((y == 1).sum())
    spw    = round(n_neg / n_pos, 2)
    summary_rows.append({
        'Disease'          : disease,
        'Samples'          : len(X),
        'Features'         : X.shape[1],
        'Positive (n)'     : n_pos,
        'Negative (n)'     : n_neg,
        'Positive rate'    : f'{y.mean():.3f}',
        'scale_pos_weight' : spw,
    })

df_summary = pd.DataFrame(summary_rows).set_index('Disease')
print(df_summary.to_string())

## 2. Framework Selection

| Framework | Role | Justification |
|---|---|---|
| **Scikit-Learn** | Baseline models + MLP neural network | Unified API; `LogisticRegression`, `RandomForestClassifier`, `MLPClassifier` all interoperate with the same `Pipeline` and `RandomizedSearchCV` interface |
| **XGBoost** | Primary model | State-of-the-art performance on tabular clinical data; native support for class-imbalance via `scale_pos_weight`; produces SHAP-compatible tree structures for Phase 5 explainability |

PyTorch and TensorFlow were considered but rejected: the dataset (5 311 samples, 8 features) is firmly in the tabular regime where gradient-boosted trees consistently outperform deep neural networks, and the interpretability requirement favours tree-based SHAP over activation-based saliency.

## 3. Model Architectures

Four model classes are defined for each of the three disease targets.

| # | Model | Role |
|---|---|---|
| M1 | Logistic Regression | Linear baseline |
| M2 | Random Forest | Non-linear ensemble baseline |
| M3 | XGBoost | Primary model |
| M4 | MLP Neural Network | Deep-learning baseline |

---

### M1 — Logistic Regression

| Component | Choice | Details |
|---|---|---|
| **Objective** | Binary classification | `multi_class='ovr'` (one vs rest) |
| **Loss function** | Log loss (binary cross-entropy) | $\mathcal{L} = -[y\log\hat{p} + (1-y)\log(1-\hat{p})]$ |
| **Optimiser** | L-BFGS | Quasi-Newton; efficient for small-to-medium datasets |
| **Regularisation** | L2 (Ridge) | Penalty $\lambda\|\mathbf{w}\|^2$ controlled by `C = 1/λ` |
| **Class imbalance** | `class_weight='balanced'` | Inverse-frequency sample weighting |

In [ ]:
# ── M1: Logistic Regression configuration ────────────────────────────────────
# One instance per disease; class_weight='balanced' handles imbalance

LR_PARAMS = dict(
    class_weight = 'balanced',
    max_iter     = 1000,
    solver       = 'lbfgs',
    C            = 1.0,          # inverse regularisation strength
    random_state = RANDOM_STATE,
)

lr_configs = {d: LogisticRegression(**LR_PARAMS) for d in DISEASES}

for name, clf in lr_configs.items():
    print(f'[{name}]  {clf}')

### M2 — Random Forest

| Component | Choice | Details |
|---|---|---|
| **Ensemble method** | Bagging + random feature subsets | `n_estimators` trees grown in parallel |
| **Split criterion** | Gini impurity | $G = 1 - \sum_k p_k^2$; computationally lighter than entropy |
| **Loss function** | Implicit; Gini is minimised at each node | Not a global differentiable loss |
| **Optimiser** | Greedy recursive partitioning | No gradient descent; each tree is grown independently |
| **Regularisation** | `max_depth`, `min_samples_leaf` | Constrains tree complexity |
| **Class imbalance** | `class_weight='balanced'` | Inverse-frequency weighting |

In [ ]:
# ── M2: Random Forest configuration ──────────────────────────────────────────

RF_PARAMS = dict(
    n_estimators   = 300,
    max_depth      = 10,
    min_samples_leaf = 5,
    max_features   = 'sqrt',     # √p features per split (standard for classification)
    class_weight   = 'balanced',
    n_jobs         = -1,
    random_state   = RANDOM_STATE,
)

rf_configs = {d: RandomForestClassifier(**RF_PARAMS) for d in DISEASES}

for name, clf in rf_configs.items():
    print(f'[{name}]  n_estimators={clf.n_estimators}  max_depth={clf.max_depth}')

### M3 — XGBoost (Primary Model)

XGBoost implements **gradient-boosted decision trees (GBDT)**.  
Each iteration adds a new weak learner $f_t$ that minimises the regularised second-order Taylor expansion of the loss:

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n}\left[g_i f_t(\mathbf{x}_i) + \frac{1}{2}h_i f_t^2(\mathbf{x}_i)\right] + \Omega(f_t)$$

where $g_i = \partial_{\hat{y}^{(t-1)}} \ell(y_i, \hat{y}^{(t-1)})$ and $h_i = \partial^2_{\hat{y}^{(t-1)}} \ell(y_i, \hat{y}^{(t-1)})$.

| Component | Choice | Details |
|---|---|---|
| **Objective** | `binary:logistic` | Outputs probability via sigmoid; trained with log loss |
| **Loss function** | Log loss (binary cross-entropy) | Same functional form as M1 but optimised with 2nd-order gradients |
| **Optimiser** | Gradient boosting (Newton steps) | Adds $\eta \cdot f_t$ to the ensemble each round |
| **Learning rate** | `eta` / `learning_rate` | Shrinks each tree's contribution; prevents overfitting |
| **Regularisation** | L1 (`reg_alpha`) + L2 (`reg_lambda`) + `gamma` (min split gain) | Triple regularisation makes XGBoost robust on small datasets |
| **Class imbalance** | `scale_pos_weight = neg/pos` | Upweights positive class gradient; computed per disease |

In [ ]:
# ── M3: XGBoost configuration (one per disease) ───────────────────────────────
# scale_pos_weight is disease-specific; all other params are shared defaults
# that will be refined by hyperparameter search in Phase 4.

SCALE_POS_WEIGHTS = {
    'diabetes'      : 18.80,   # 5 038 neg / 273 pos  ≈ 18.4×  (rounded)
    'heart_disease' :  5.80,   #   789 neg / 782 pos  ≈ 5.8×
    'hypertension'  :  1.64,   # 3 300 neg / 2 011 pos ≈ 1.6×
}

XGB_BASE_PARAMS = dict(
    objective         = 'binary:logistic',
    eval_metric       = 'logloss',
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 5,
    min_child_weight  = 3,
    subsample         = 0.80,
    colsample_bytree  = 0.80,
    gamma             = 0.10,
    reg_alpha         = 0.10,
    reg_lambda        = 1.0,
    n_jobs            = -1,
    random_state      = RANDOM_STATE,
)

xgb_configs = {
    d: XGBClassifier(**XGB_BASE_PARAMS, scale_pos_weight=SCALE_POS_WEIGHTS[d])
    for d in DISEASES
}

for name, clf in xgb_configs.items():
    print(f'[{name}]  scale_pos_weight={SCALE_POS_WEIGHTS[name]}  '
          f'n_estimators={clf.n_estimators}  lr={clf.learning_rate}  '
          f'max_depth={clf.max_depth}')

### M4 — MLP Neural Network

A fully-connected feed-forward network implemented via `sklearn.neural_network.MLPClassifier`.

#### Architecture

```
Input Layer       →   8 neurons  (one per scaled feature)
Hidden Layer 1    →  128 neurons  (ReLU)
Hidden Layer 2    →   64 neurons  (ReLU)
Output Layer      →    1 neuron   (Sigmoid → probability)
```

| Component | Choice | Details |
|---|---|---|
| **Hidden layers** | (128, 64) | Two layers; deeper than (64,) but avoids over-parameterisation on 5 311 samples |
| **Activation** | ReLU | $f(z) = \max(0, z)$; avoids vanishing gradient; faster than tanh |
| **Loss function** | Log loss (binary cross-entropy) | `sklearn` uses `log_loss` internally for binary targets |
| **Optimiser** | Adam | Adaptive moment estimation; `learning_rate_init=0.001`; handles sparse gradients well |
| **Regularisation** | L2 weight decay (`alpha`) | Penalises large weights; controls generalisation |
| **Early stopping** | Enabled | Monitors validation loss; stops when no improvement for 20 iterations |
| **Class imbalance** | Sample weights in `fit()` | Passed as `sample_weight` array during Phase 4 training |

In [ ]:
# ── M4: MLP Neural Network configuration ─────────────────────────────────────

MLP_PARAMS = dict(
    hidden_layer_sizes   = (128, 64),  # architecture: 8 → 128 → 64 → 1
    activation           = 'relu',
    solver               = 'adam',
    alpha                = 0.001,      # L2 regularisation
    batch_size           = 64,
    learning_rate        = 'adaptive', # halve lr when loss stops improving
    learning_rate_init   = 0.001,
    max_iter             = 500,
    early_stopping       = True,
    validation_fraction  = 0.10,
    n_iter_no_change     = 20,
    random_state         = RANDOM_STATE,
)

mlp_configs = {d: MLPClassifier(**MLP_PARAMS) for d in DISEASES}

for name, clf in mlp_configs.items():
    print(f'[{name}]  layers={clf.hidden_layer_sizes}  '
          f'activation={clf.activation}  solver={clf.solver}  '
          f'alpha={clf.alpha}')

## 4. Architecture Diagram — MLP

In [ ]:
# ── MLP architecture diagram ──────────────────────────────────────────────────

FEATURE_NAMES = ['age', 'bmi', 'glucose', 'blood_pressure',
                 'insulin', 'cholesterol', 'source_pima', 'source_uci_heart']

LAYERS = [
    ('Input\n(8 features)',  8,  '#4C9BE8'),
    ('Hidden 1\n128 · ReLU', 8,  '#5FBF7A'),
    ('Hidden 2\n64 · ReLU',  8,  '#5FBF7A'),
    ('Output\n1 · Sigmoid',  1,  '#E85D5D'),
]

fig, ax = plt.subplots(figsize=(11, 5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('MLP Architecture  ·  8 → 128 → 64 → 1', fontsize=13, fontweight='bold', pad=12)

n_layers = len(LAYERS)
xs = np.linspace(0.10, 0.90, n_layers)
max_neurons_display = 8

for col_idx, (label, n_neurons, color) in enumerate(LAYERS):
    x = xs[col_idx]
    display_n = min(n_neurons, max_neurons_display)
    ys = np.linspace(0.15, 0.85, display_n)

    # draw neurons
    for y in ys:
        circle = plt.Circle((x, y), 0.030, color=color, zorder=3, alpha=0.88)
        ax.add_patch(circle)

    # draw connections to next layer
    if col_idx < n_layers - 1:
        next_label, next_n, _ = LAYERS[col_idx + 1]
        next_x = xs[col_idx + 1]
        next_display = min(next_n, max_neurons_display)
        next_ys = np.linspace(0.15, 0.85, next_display)
        for y_src in ys:
            for y_dst in next_ys:
                ax.plot([x + 0.030, next_x - 0.030], [y_src, y_dst],
                        color='#aaaaaa', linewidth=0.4, alpha=0.5, zorder=1)

    # layer label
    ax.text(x, 0.03, label, ha='center', va='bottom', fontsize=9,
            fontweight='bold', color='#333333')

# legend
handles = [
    mpatches.Patch(color='#4C9BE8', label='Input layer'),
    mpatches.Patch(color='#5FBF7A', label='Hidden layers (ReLU)'),
    mpatches.Patch(color='#E85D5D', label='Output layer (Sigmoid)'),
]
ax.legend(handles=handles, loc='upper right', fontsize=9, framealpha=0.85)

plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '12_mlp_architecture.png'))
plt.show()
print('Diagram saved.')

## 5. Hyperparameter Tuning Process

### Strategy

| Step | Method | Rationale |
|---|---|---|
| Search algorithm | `RandomizedSearchCV` | More efficient than grid search for continuous distributions; 40 iterations ≈ 90 % of full-grid coverage at < 10 % of the cost |
| Cross-validation | `StratifiedKFold(n_splits=5)` | Preserves class ratio in each fold; critical given severe imbalance for diabetes |
| Scoring metric | `roc_auc` | Threshold-free; robust under class imbalance; preferred over accuracy |
| `n_iter` | 40 (XGBoost), 16 (MLP) | Balances exploration and compute time |

---

### 5a. XGBoost — Hyperparameter Search Space

In [ ]:
from scipy.stats import uniform, randint

XGB_PARAM_DISTRIBUTIONS = {
    'learning_rate'    : uniform(0.01, 0.29),    # U[0.01, 0.30]
    'n_estimators'     : randint(100, 501),       # {100 … 500}
    'max_depth'        : randint(3, 7),           # {3, 4, 5, 6}
    'min_child_weight' : randint(1, 6),           # {1 … 5}
    'subsample'        : uniform(0.60, 0.40),     # U[0.60, 1.00]
    'colsample_bytree' : uniform(0.60, 0.40),     # U[0.60, 1.00]
    'gamma'            : uniform(0.00, 0.30),     # U[0.00, 0.30]
    'reg_alpha'        : uniform(0.00, 0.50),     # L1 penalty
    'reg_lambda'       : uniform(0.50, 1.50),     # L2 penalty
}

# Display the search space as a summary table
rows = []
ranges = {
    'learning_rate'    : '0.01 – 0.30',
    'n_estimators'     : '100 – 500',
    'max_depth'        : '3 – 6',
    'min_child_weight' : '1 – 5',
    'subsample'        : '0.60 – 1.00',
    'colsample_bytree' : '0.60 – 1.00',
    'gamma'            : '0.00 – 0.30',
    'reg_alpha'        : '0.00 – 0.50',
    'reg_lambda'       : '0.50 – 2.00',
}
dist_types = {
    'learning_rate'    : 'Continuous uniform',
    'n_estimators'     : 'Discrete uniform',
    'max_depth'        : 'Discrete uniform',
    'min_child_weight' : 'Discrete uniform',
    'subsample'        : 'Continuous uniform',
    'colsample_bytree' : 'Continuous uniform',
    'gamma'            : 'Continuous uniform',
    'reg_alpha'        : 'Continuous uniform',
    'reg_lambda'       : 'Continuous uniform',
}
roles = {
    'learning_rate'    : 'Step size per boosting round',
    'n_estimators'     : 'Number of boosting rounds (trees)',
    'max_depth'        : 'Maximum depth of each tree',
    'min_child_weight' : 'Min sum of instance weight in a leaf',
    'subsample'        : 'Fraction of rows sampled per tree',
    'colsample_bytree' : 'Fraction of features sampled per tree',
    'gamma'            : 'Min loss reduction to make a split',
    'reg_alpha'        : 'L1 regularisation on leaf weights',
    'reg_lambda'       : 'L2 regularisation on leaf weights',
}

for param in XGB_PARAM_DISTRIBUTIONS:
    rows.append({
        'Parameter'    : param,
        'Distribution' : dist_types[param],
        'Range'        : ranges[param],
        'Role'         : roles[param],
    })

df_xgb_space = pd.DataFrame(rows).set_index('Parameter')
print('XGBoost Hyperparameter Search Space')
print('=' * 80)
print(df_xgb_space.to_string())

### 5b. MLP — Hyperparameter Search Space

In [ ]:
MLP_PARAM_GRID = {
    'hidden_layer_sizes' : [(64,), (128,), (64, 32), (128, 64), (128, 64, 32)],
    'activation'         : ['relu', 'tanh'],
    'alpha'              : [1e-4, 1e-3, 1e-2],
    'learning_rate_init' : [0.001, 0.01],
    'batch_size'         : [32, 64, 128],
}

# Total combinations
from math import prod
total = prod(len(v) for v in MLP_PARAM_GRID.values())
print(f'Total MLP grid combinations: {total}')
print(f'RandomizedSearchCV will sample: 16 / {total}  ({16/total*100:.1f}%)\n')

mlp_rows = [
    {'Parameter': 'hidden_layer_sizes', 'Candidates': '(64,)  (128,)  (64,32)  (128,64)  (128,64,32)',
     'Role': 'Network depth and width'},
    {'Parameter': 'activation',         'Candidates': 'relu · tanh',
     'Role': 'Non-linearity of hidden neurons'},
    {'Parameter': 'alpha',              'Candidates': '0.0001 · 0.001 · 0.01',
     'Role': 'L2 weight-decay coefficient'},
    {'Parameter': 'learning_rate_init', 'Candidates': '0.001 · 0.01',
     'Role': 'Initial Adam step size'},
    {'Parameter': 'batch_size',         'Candidates': '32 · 64 · 128',
     'Role': 'Mini-batch size per gradient update'},
]
df_mlp_space = pd.DataFrame(mlp_rows).set_index('Parameter')
print('MLP Hyperparameter Search Space')
print('=' * 70)
print(df_mlp_space.to_string())

## 6. Final Hyperparameter Selections (Pre-Search Defaults)

The table below documents the **default configurations** used as the starting point  
before `RandomizedSearchCV` is executed in Phase 4.  
After the search, this table will be updated with the best parameters found per disease.

### 6a. XGBoost — Default vs Tuning Rationale

| Parameter | Default Value | Tuning Rationale |
|---|---|---|
| `learning_rate` | 0.05 | Low shrinkage reduces overfitting; search will determine if faster rates are viable |
| `n_estimators` | 300 | Enough rounds to converge; early stopping in Phase 4 will find the true optimum |
| `max_depth` | 5 | Moderate depth; shallower trees generalise better on 5 k samples |
| `min_child_weight` | 3 | Higher values make the model more conservative; important for 5 % positive-rate diabetes |
| `subsample` | 0.80 | 20 % row-dropout per tree; reduces variance |
| `colsample_bytree` | 0.80 | 20 % feature-dropout per tree; encourages feature diversity |
| `gamma` | 0.10 | Non-zero minimum split gain prunes unnecessary leaves |
| `reg_alpha` | 0.10 | Light L1 encourages sparse trees |
| `reg_lambda` | 1.0 | XGBoost default L2; provides baseline weight regularisation |
| `scale_pos_weight` | per-disease | Computed as neg/pos ratio; non-negotiable for class-imbalanced diseases |

### 6b. MLP — Default vs Tuning Rationale

| Parameter | Default Value | Tuning Rationale |
|---|---|---|
| `hidden_layer_sizes` | (128, 64) | Two layers chosen as baseline; single-layer (64,) may suffice given feature count |
| `activation` | relu | ReLU is default; tanh will be compared in search |
| `solver` | adam | Fixed; L-BFGS impractical for mini-batch; SGD too sensitive |
| `alpha` | 0.001 | Moderate L2; too small risks overfitting, too large underfitting |
| `learning_rate_init` | 0.001 | Standard Adam default; 0.01 tested in search |
| `batch_size` | 64 | Balances gradient noise and stability; 32 and 128 also searched |
| `max_iter` | 500 | Upper limit; early stopping will halt training before this |
| `early_stopping` | True | Monitors 10 % held-out validation loss; stops after 20 non-improving iterations |

## 7. Visualisation — Hyperparameter Search Space

In [ ]:
# ── Visualise class-imbalance ratios and scale_pos_weight per disease ──────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: class balance stacked bar
disease_labels = ['Diabetes\n(5.0%)', 'Heart Disease\n(14.7%)', 'Hypertension\n(37.9%)']
pos_rates      = [0.050, 0.147, 0.379]
neg_rates      = [1 - p for p in pos_rates]
colors         = ['#E85D5D', '#4C9BE8']

bars_neg = axes[0].bar(disease_labels, neg_rates, color='#4C9BE8', label='Negative')
bars_pos = axes[0].bar(disease_labels, pos_rates,  bottom=neg_rates,
                        color='#E85D5D', label='Positive')
axes[0].set_ylabel('Proportion')
axes[0].set_title('Class Distribution per Disease')
axes[0].legend(loc='upper right')
axes[0].set_ylim(0, 1.05)

# annotate positive rate
for bar, pr in zip(bars_pos, pos_rates):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_y() + bar.get_height() / 2,
                 f'{pr:.1%}', ha='center', va='center', color='white',
                 fontsize=9, fontweight='bold')

# Panel 2: scale_pos_weight bar
spw_vals = list(SCALE_POS_WEIGHTS.values())
spw_keys = [k.replace('_', ' ').title() for k in SCALE_POS_WEIGHTS.keys()]
bar_objs = axes[1].bar(spw_keys, spw_vals,
                        color=['#E85D5D', '#FFA040', '#5FBF7A'], edgecolor='white')
axes[1].bar_label(bar_objs, fmt='%.1f×', padding=3)
axes[1].set_ylabel('scale_pos_weight  (neg / pos)')
axes[1].set_title('XGBoost scale_pos_weight per Disease')
axes[1].set_ylim(0, max(spw_vals) * 1.2)

plt.suptitle('Class Imbalance & Derived Hyperparameters', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '13_class_imbalance_spw.png'))
plt.show()

In [ ]:
# ── Visualise XGBoost continuous parameter search ranges ──────────────────────

continuous_params = {
    'learning_rate'    : (0.01, 0.30),
    'subsample'        : (0.60, 1.00),
    'colsample_bytree' : (0.60, 1.00),
    'gamma'            : (0.00, 0.30),
    'reg_alpha'        : (0.00, 0.50),
    'reg_lambda'       : (0.50, 2.00),
}
defaults = {
    'learning_rate'    : 0.05,
    'subsample'        : 0.80,
    'colsample_bytree' : 0.80,
    'gamma'            : 0.10,
    'reg_alpha'        : 0.10,
    'reg_lambda'       : 1.00,
}

params   = list(continuous_params.keys())
n_params = len(params)

fig, ax = plt.subplots(figsize=(10, 4))
for i, p in enumerate(params):
    lo, hi = continuous_params[p]
    ax.barh(i, hi - lo, left=lo, height=0.45,
            color='#4C9BE8', alpha=0.55, label='Search range' if i == 0 else '')
    ax.scatter(defaults[p], i, color='#E85D5D', zorder=5, s=80,
               label='Default value' if i == 0 else '')

ax.set_yticks(range(n_params))
ax.set_yticklabels(params)
ax.set_xlabel('Parameter value')
ax.set_title('XGBoost — Continuous Hyperparameter Search Ranges\n(red dot = default starting point)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '14_xgb_search_ranges.png'))
plt.show()

## 8. Serialise Configurations

Model configurations (as parameter dictionaries) and the `scale_pos_weight` mapping  
are saved to `data/processed/` so Phase 4 can load them without re-defining them.

In [ ]:
# ── Save parameter dictionaries ───────────────────────────────────────────────

config = {
    'random_state'        : RANDOM_STATE,
    'test_size'           : TEST_SIZE,
    'cv_folds'            : CV_FOLDS,
    'scale_pos_weight'    : SCALE_POS_WEIGHTS,
    'lr_params'           : LR_PARAMS,
    'rf_params'           : RF_PARAMS,
    'xgb_base_params'     : XGB_BASE_PARAMS,
    'mlp_params'          : MLP_PARAMS,
    'xgb_search_n_iter'   : 40,
    'mlp_search_n_iter'   : 16,
    'search_scoring'      : 'roc_auc',
}

# Convert tuple keys that are not JSON-serialisable
config_json = config.copy()
config_json['mlp_params'] = dict(config['mlp_params'])
config_json['mlp_params']['hidden_layer_sizes'] = list(
    config['mlp_params']['hidden_layer_sizes']
)

out_path = os.path.join(PROC_DIR, 'model_configs.json')
with open(out_path, 'w') as fh:
    json.dump(config_json, fh, indent=2)
print(f'Configurations saved → {out_path}')

# Save unfitted sklearn estimator objects (with params embedded) for Phase 4
joblib.dump(lr_configs,  os.path.join(MODEL_DIR, 'lr_configs.pkl'))
joblib.dump(rf_configs,  os.path.join(MODEL_DIR, 'rf_configs.pkl'))
joblib.dump(xgb_configs, os.path.join(MODEL_DIR, 'xgb_configs.pkl'))
joblib.dump(mlp_configs, os.path.join(MODEL_DIR, 'mlp_configs.pkl'))

print('Unfitted model objects saved to', MODEL_DIR)
print('\nSaved files:')
for f in os.listdir(MODEL_DIR):
    print(f'  {f}')

## 9. Phase 3 Summary

| Item | Decision |
|---|---|
| **Framework** | Scikit-Learn 1.6 + XGBoost 3.0 |
| **Models defined** | M1 Logistic Regression · M2 Random Forest · M3 XGBoost · M4 MLP |
| **Primary model** | M3 XGBoost (`binary:logistic`, log-loss, gradient-boosting optimiser) |
| **Neural network** | M4 MLP  8 → 128 → 64 → 1  ·  ReLU  ·  Adam  ·  log-loss |
| **Imbalance strategy** | `scale_pos_weight` (XGBoost) · `class_weight='balanced'` (LR, RF) · sample weights (MLP) |
| **Tuning algorithm** | `RandomizedSearchCV`  ·  `n_iter=40` (XGB) / `16` (MLP)  ·  `cv=5`  ·  scored by `roc_auc` |
| **Outputs** | `model_configs.json` · `*_configs.pkl` (unfitted) · architecture diagram |
| **Next phase** | Phase 4 — Training, cross-validation, and evaluation |